# 04 - Compare with `lactationcurve` and Plan a Migration

Bovi's existing `best_predict_method` is a narrower milk-only model. BESTPRED uses
additional identity, herd, component and collection information and returns many more
metrics. The comparison below aligns test days and units; it is an evaluation tool,
not an equality test.

In [1]:
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "packages/models/bestpred").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from a Bovi repository checkout")


ROOT = find_repo_root()
PACKAGE_ROOT = ROOT / "packages/models/bestpred"
FIXTURES = PACKAGE_ROOT / "tests/fixtures"
PARAMETERS = FIXTURES / "source11_current/bestpred.par"

In [2]:
import pandas as pd
from lactationcurve.characteristics.best_predict import best_predict_method

from bestpred import predict_dataframe

test_days = pd.read_csv(
    PACKAGE_ROOT / "notebooks/data/example_test_days.csv",
    dtype={"BirthDate": str, "FreshDate": str},
)
bestpred_result = predict_dataframe(test_days, PARAMETERS)

LB_PER_KG = 2.2046226218487757
bovi_input = test_days[["TestId", "DaysInMilk", "MilkingYield"]].copy()
bovi_input["MilkingYield"] = bovi_input["MilkingYield"] / LB_PER_KG
bovi_result = best_predict_method(bovi_input)

comparison = bestpred_result[["TestId", "MilkYield305"]].merge(
    bovi_result[["TestId", "LactationMilkYield"]], on="TestId", how="left"
)
comparison["BestpredMilk305Kg"] = comparison["MilkYield305"] / LB_PER_KG
comparison["DeltaKg"] = comparison["BestpredMilk305Kg"] - comparison["LactationMilkYield"]
comparison[["TestId", "BestpredMilk305Kg", "LactationMilkYield", "DeltaKg"]].round(2)

,TestId,BestpredMilk305Kg,LactationMilkYield,DeltaKg
0,cow-42-l2,8787.30,8478.58,308.72
1,cow-77-l1,9247.55,7745.85,1501.71


Differences are expected: the models have different standard curves, covariance logic,
adjustment data and contracts. A replacement decision needs an agreed validation dataset
and acceptance criteria, not merely a zero delta on these demonstrations.

## A backwards-compatible wrapper shape

In [3]:
def bestpred_as_lactationcurve_output(
    dataframe: pd.DataFrame,
    parameter_path: Path,
) -> pd.DataFrame:
    # Return the old public output columns while using BESTPRED internally.

    predicted = predict_dataframe(dataframe, parameter_path)
    return pd.DataFrame(
        {
            "TestId": predicted["TestId"],
            # This parameter fixture emits pounds; the current Bovi API emits kilograms.
            "LactationMilkYield": predicted["MilkYield305"] / LB_PER_KG,
        }
    )


bestpred_as_lactationcurve_output(test_days, PARAMETERS).round(2)

,TestId,LactationMilkYield
0,cow-42-l2,8787.30
1,cow-77-l1,9247.55


This wrapper demonstrates output compatibility only. A production wrapper must own the
parameter set and unit contract explicitly, and must decide how missing herd/component
facts are obtained rather than silently inventing them.

## Capability and readiness review

In [4]:
review = pd.DataFrame(
    [
        ("Milk-only 305-day estimate", "Both", "Choose using validation evidence"),
        ("Fat/protein/SCS", "BESTPRED", "Available in package output"),
        ("DCR/reliability/persistency", "BESTPRED", "Available in package output"),
        ("Minimal three-column input", "lactationcurve", "BESTPRED needs richer facts"),
        ("Canonical FDD Lactation/TestDay", "Missing", "Define upstream first"),
        ("Parameter ownership/versioning", "Missing", "Required for reproducible service output"),
        ("Production API/backend routing", "Missing", "No endpoint calls BESTPRED today"),
        ("Unit contract at service boundary", "Missing", "Must be explicit and tested"),
        ("Acceptance thresholds and benchmark", "Missing", "Agree by breed/parity/data pattern"),
        ("Source 12 parser", "Missing", "Only needed if USDA master input remains required"),
        (
            "Fortran compatibility quirks",
            "Documented",
            "Separate compatibility from desired behavior",
        ),
        ("Performance/operational monitoring", "Missing", "Benchmark batches and report failures"),
    ],
    columns=["Topic", "Current state", "Next decision/work"],
)
review

,Topic,Current state,Next decision/work
0,Milk-only 305-day estimate,Both,Choose using validation evidence
1,Fat/protein/SCS,BESTPRED,Available in package output
2,DCR/reliability/persistency,BESTPRED,Available in package output
3,Minimal three-column input,lactationcurve,BESTPRED needs richer facts
4,Canonical FDD Lactation/TestDay,Missing,Define upstream first
5,Parameter ownership/versioning,Missing,Required for reproducible service output
6,Production API/backend routing,Missing,No endpoint calls BESTPRED today
7,Unit contract at service boundary,Missing,Must be explicit and tested
8,Acceptance thresholds and benchmark,Missing,Agree by breed/parity/data pattern
9,Source 12 parser,Missing,Only needed if USDA master input remains required


## Recommended migration sequence

1. Define unit-explicit FDD Lactation, TestDay, herd-baseline and prediction models.
2. Freeze/version the approved BESTPRED parameter assets and conversion policy.
3. Validate both methods on representative breed, parity and recording patterns.
4. Add a typed internal BESTPRED service boundary and keep the old DataFrame response wrapper.
5. Shadow-run and monitor deltas before changing the production default.
6. Retire the old method only after downstream consumers and acceptance criteria pass.